In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder \
.config("spark.port.ui", 0) \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse") \
.enableHiveSupport() \
.master("yarn") \
.getOrCreate()

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

clickstream_df = spark.range(50000) \
    .withColumn("user_id", (F.rand() * 100).cast("int")) \
    .withColumn("click_timestamp", F.to_timestamp(F.lit("2025-08-03")) + F.expr("rand() * 10000 * interval 1 minute")) \
    .withColumn("page_url", F.expr("case when rand() < 0.2 then '/home' when rand() < 0.6 and rand() > 0.2 then '/catalog' else '/checkout' end"))

In [4]:
clickstream_df.show(truncate=False)

+---+-------+--------------------------+---------+
|id |user_id|click_timestamp           |page_url |
+---+-------+--------------------------+---------+
|0  |93     |2025-08-05 19:11:52.34692 |/catalog |
|1  |6      |2025-08-06 08:44:06.457656|/home    |
|2  |24     |2025-08-07 15:53:29.727165|/checkout|
|3  |93     |2025-08-08 23:45:03.318368|/catalog |
|4  |58     |2025-08-04 12:24:22.242087|/checkout|
|5  |18     |2025-08-05 05:29:38.091631|/catalog |
|6  |45     |2025-08-07 13:20:38.901884|/catalog |
|7  |74     |2025-08-06 12:38:15.108153|/checkout|
|8  |4      |2025-08-08 13:47:33.394161|/catalog |
|9  |48     |2025-08-05 10:23:21.907673|/checkout|
|10 |40     |2025-08-08 03:15:17.455858|/catalog |
|11 |69     |2025-08-07 20:51:29.721432|/checkout|
|12 |72     |2025-08-06 14:10:01.165141|/home    |
|13 |24     |2025-08-09 14:32:53.079902|/home    |
|14 |20     |2025-08-07 01:56:30.596391|/checkout|
|15 |48     |2025-08-08 20:36:12.396646|/checkout|
|16 |47     |2025-08-04 07:02:0

In [5]:
from pyspark.sql import Window

window_spec = Window.partitionBy("user_id").orderBy(F.col("click_timestamp").asc())
last_active_df = clickstream_df.withColumn("last_timestamp", F.lag("click_timestamp", 1).over(window_spec))
activity_diff_df = last_active_df.withColumn("timestamp_gap_seconds", F.col("click_timestamp").cast("long")-F.col("last_timestamp").cast("long"))
activity_diff_df.show(truncate=False)

+-----+-------+--------------------------+---------+--------------------------+---------------------+
|id   |user_id|click_timestamp           |page_url |last_timestamp            |timestamp_gap_seconds|
+-----+-------+--------------------------+---------+--------------------------+---------------------+
|33752|31     |2025-08-03 01:02:26.11617 |/catalog |null                      |null                 |
|19473|31     |2025-08-03 01:22:50.554484|/home    |2025-08-03 01:02:26.11617 |1224                 |
|42419|31     |2025-08-03 01:23:56.666181|/catalog |2025-08-03 01:22:50.554484|66                   |
|16221|31     |2025-08-03 01:36:31.921406|/catalog |2025-08-03 01:23:56.666181|755                  |
|24712|31     |2025-08-03 02:04:06.270615|/checkout|2025-08-03 01:36:31.921406|1655                 |
|48951|31     |2025-08-03 02:48:06.548205|/checkout|2025-08-03 02:04:06.270615|2640                 |
|18775|31     |2025-08-03 03:12:40.211685|/checkout|2025-08-03 02:48:06.548205|147

In [8]:
is_session_df = activity_diff_df.withColumn("is_new_session", 
                                            F.when(F.col("timestamp_gap_seconds") > 1800, 1) \
                                             .when(F.col("timestamp_gap_seconds").isNull(), 1) \
                                             .otherwise(0))
session_df = is_session_df.withColumn("user_session_id", F.sum("is_new_session").over(window_spec)) \
                          .withColumn("global_session_id", F.concat(F.col("user_id"), F.lit("_session_"), F.col("user_session_id")))
session_df.show(truncate=False)

+-----+-------+--------------------------+---------+--------------------------+---------------------+--------------+---------------+-----------------+
|id   |user_id|click_timestamp           |page_url |last_timestamp            |timestamp_gap_seconds|is_new_session|user_session_id|global_session_id|
+-----+-------+--------------------------+---------+--------------------------+---------------------+--------------+---------------+-----------------+
|33752|31     |2025-08-03 01:02:26.11617 |/catalog |null                      |null                 |1             |1              |31_session_1     |
|19473|31     |2025-08-03 01:22:50.554484|/home    |2025-08-03 01:02:26.11617 |1224                 |0             |1              |31_session_1     |
|42419|31     |2025-08-03 01:23:56.666181|/catalog |2025-08-03 01:22:50.554484|66                   |0             |1              |31_session_1     |
|16221|31     |2025-08-03 01:36:31.921406|/catalog |2025-08-03 01:23:56.666181|755            